<a href="https://colab.research.google.com/github/mf2056/F20AA_CW2/blob/main/Step4%265.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [2]:
!pip install -U scikeras scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 24.1 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [3]:
train = pd.read_csv("/content/drive/MyDrive/train_processed.csv")

train.head()

,text,rating,tokens,stemmed,lemmatized,text_stem,text_lemma,review_length
0,this place is terrible the people in charge ar...,2,"['place', 'terrible', 'people', 'charge', 'wor...","['place', 'terribl', 'peopl', 'charg', 'worst'...","['place', 'terrible', 'people', 'charge', 'wor...",place terribl peopl charg worst part far yeah ...,place terrible people charge worst part far ye...,97
1,terrible service and they are saying that i ne...,1,"['terrible', 'service', 'saying', 'never', 'us...","['terribl', 'servic', 'say', 'never', 'use', '...","['terrible', 'service', 'saying', 'never', 'us...",terribl servic say never use servic lie call n...,terrible service saying never used service lie...,48
2,absolutely terrible company they sent me to co...,1,"['absolutely', 'terrible', 'company', 'sent', ...","['absolut', 'terribl', 'compani', 'sent', 'col...","['absolutely', 'terrible', 'company', 'sent', ...",absolut terribl compani sent collect without a...,absolutely terrible company sent collection wi...,211
3,to find it either park in front of the tuesday...,4,"['find', 'either', 'park', 'front', 'tuesday',...","['find', 'either', 'park', 'front', 'tuesday',...","['find', 'either', 'park', 'front', 'tuesday',...",find either park front tuesday morn mall entra...,find either park front tuesday morning mall en...,66
4,mall location used their services for sedan ni...,4,"['mall', 'location', 'used', 'services', 'seda...","['mall', 'locat', 'use', 'servic', 'sedan', 'n...","['mall', 'location', 'used', 'service', 'sedan...",mall locat use servic sedan nice perhap inform...,mall location used service sedan nice perhaps ...,28


In [4]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.metrics import classification_report

In [6]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

In [5]:
train.isnull().sum()

,0
text,43
rating,0
tokens,0
stemmed,0
lemmatized,0
text_stem,105
text_lemma,105
review_length,0


In [6]:
train["text_lemma"] = train["text_lemma"].fillna("")
train["text_lemma"] = train["text_lemma"].astype(str)

train["text_stem"] = train["text_stem"].fillna("")
train["text_stem"] = train["text_stem"].astype(str)

train["text"] = train["text"].fillna("")
train["text"] = train["text"].astype(str)

X_d = train["text"]
X = train["text_lemma"]
x = train["text_stem"]
Y = train["rating"]

In [7]:
train.shape

(288000, 8)

In [8]:
train.isnull().sum()

,0
text,0
rating,0
tokens,0
stemmed,0
lemmatized,0
text_stem,0
text_lemma,0
review_length,0


In [9]:
import pandas as pd
from sklearn.model_selection import GridSearchCV

def run_gridsearch(name, pipeline, params):
    # 1. Define multiple metrics
    scoring = {
        'Accuracy': 'accuracy',
        'MacroF1': 'f1_macro'
    }

    grid = GridSearchCV(
        pipeline,
        params,
        cv=3,
        n_jobs=-1,
        verbose=1,
        scoring=scoring,
        refit='MacroF1',
        return_train_score=False
    )

    grid.fit(X, Y)

    # 2. Extract detailed results into a table
    results_df = pd.DataFrame(grid.cv_results_)

    # Select columns for the 3 folds and the means
    columns_to_show = ['params']
    # Add Accuracy columns
    columns_to_show += ['split0_test_Accuracy', 'split1_test_Accuracy', 'split2_test_Accuracy', 'mean_test_Accuracy']
    # Add F1 columns
    columns_to_show += ['split0_test_MacroF1', 'split1_test_MacroF1', 'split2_test_MacroF1', 'mean_test_MacroF1']

    report_table = results_df[columns_to_show].sort_values('mean_test_MacroF1', ascending=False)

    print(f"\n==== {name} DETAILED FOLD RESULTS ====")
    print(report_table.to_string(index=False))

    # 3. Print the absolute bests
    best_f1_idx = results_df['mean_test_MacroF1'].idxmax()
    best_acc_idx = results_df['mean_test_Accuracy'].idxmax()

    print(f"\n--- {name} Summary ---")
    print(f"Best Params (for F1): {grid.best_params_}")
    print(f"Best Mean Macro-F1: {results_df.loc[best_f1_idx, 'mean_test_MacroF1']:.4f}")
    print(f"Best Mean Accuracy: {results_df.loc[best_acc_idx, 'mean_test_Accuracy']:.4f}")

    return grid

In [49]:
nb_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", MultinomialNB())
])

nb_params = {
    "clf__alpha": [0.1, 0.5, 1.0],
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__max_features": [10000, 20000, 50000]
}

nb_grid = run_gridsearch(
    "Naive Bayes",
    nb_pipeline,
    nb_params
)

Fitting 3 folds for each of 18 candidates, totalling 54 fits

==== Naive Bayes DETAILED FOLD RESULTS ====
                                                                         params  split0_test_Accuracy  split1_test_Accuracy  split2_test_Accuracy  mean_test_Accuracy  split0_test_MacroF1  split1_test_MacroF1  split2_test_MacroF1  mean_test_MacroF1
{'clf__alpha': 0.1, 'tfidf__max_features': 50000, 'tfidf__ngram_range': (1, 2)}              0.637219              0.636490              0.635531            0.636413             0.465647             0.465948             0.465692           0.465763
{'clf__alpha': 0.1, 'tfidf__max_features': 20000, 'tfidf__ngram_range': (1, 2)}              0.635490              0.634865              0.634031            0.634795             0.455449             0.456240             0.457453           0.456381
{'clf__alpha': 0.5, 'tfidf__max_features': 20000, 'tfidf__ngram_range': (1, 2)}              0.635573              0.634500              0.634583     

In [51]:
lr_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LogisticRegression(max_iter=1000))
])

lr_params = {
    "clf__C": [0.5, 1, 3],
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__max_features": [10000, 20000, 50000]
}

lr_grid = run_gridsearch(
    "Logistic Regression",
    lr_pipeline,
    lr_params
)

Fitting 3 folds for each of 18 candidates, totalling 54 fits

==== Logistic Regression DETAILED FOLD RESULTS ====
                                                                     params  split0_test_Accuracy  split1_test_Accuracy  split2_test_Accuracy  mean_test_Accuracy  split0_test_MacroF1  split1_test_MacroF1  split2_test_MacroF1  mean_test_MacroF1
  {'clf__C': 3, 'tfidf__max_features': 20000, 'tfidf__ngram_range': (1, 2)}              0.647729              0.647885              0.648146            0.647920             0.510176             0.509683             0.511105           0.510321
  {'clf__C': 3, 'tfidf__max_features': 10000, 'tfidf__ngram_range': (1, 2)}              0.651542              0.649958              0.650062            0.650521             0.510756             0.508212             0.511439           0.510136
  {'clf__C': 3, 'tfidf__max_features': 50000, 'tfidf__ngram_range': (1, 2)}              0.646333              0.646010              0.646323            0

In [56]:
svm_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
    )),
    ("clf", LinearSVC())
])

svm_params = {
    "clf__C": [0.5, 1, 2],
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__max_features": [10000, 20000, 50000]
}

svm_grid = run_gridsearch(
    "SVM",
    svm_pipeline,
    svm_params
)

Fitting 3 folds for each of 18 candidates, totalling 54 fits

==== SVM DETAILED FOLD RESULTS ====
                                                                     params  split0_test_Accuracy  split1_test_Accuracy  split2_test_Accuracy  mean_test_Accuracy  split0_test_MacroF1  split1_test_MacroF1  split2_test_MacroF1  mean_test_MacroF1
  {'clf__C': 2, 'tfidf__max_features': 20000, 'tfidf__ngram_range': (1, 2)}              0.636646              0.635865              0.636083            0.636198             0.489704             0.489794             0.493818           0.491105
{'clf__C': 0.5, 'tfidf__max_features': 50000, 'tfidf__ngram_range': (1, 2)}              0.644823              0.643312              0.643302            0.643813             0.489808             0.489903             0.493333           0.491014
  {'clf__C': 1, 'tfidf__max_features': 50000, 'tfidf__ngram_range': (1, 2)}              0.632250              0.631396              0.631490            0.631712         

In [18]:
from sklearn.linear_model import SGDClassifier

sgd_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()), # Parameters handled by grid
    ("clf", SGDClassifier(max_iter=1000, tol=1e-3, random_state=25))
])

sgd_params = {
    "clf__alpha": [0.0001, 0.001, 0.01],
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__max_features": [10000,20000, 50000]
}

sgd_grid = run_gridsearch("SGD Classifier", sgd_pipeline, sgd_params)

Fitting 3 folds for each of 18 candidates, totalling 54 fits

==== SGD Classifier DETAILED FOLD RESULTS ====
                                                                            params  split0_test_Accuracy  split1_test_Accuracy  split2_test_Accuracy  mean_test_Accuracy  split0_test_MacroF1  split1_test_MacroF1  split2_test_MacroF1  mean_test_MacroF1
{'clf__alpha': 0.0001, 'tfidf__max_features': 10000, 'tfidf__ngram_range': (1, 2)}              0.635687              0.633729              0.633042            0.634153             0.403259             0.402127             0.403054           0.402813
{'clf__alpha': 0.0001, 'tfidf__max_features': 20000, 'tfidf__ngram_range': (1, 2)}              0.634667              0.632823              0.632656            0.633382             0.399555             0.399090             0.400321           0.399656
{'clf__alpha': 0.0001, 'tfidf__max_features': 10000, 'tfidf__ngram_range': (1, 1)}              0.633646              0.630354            

In [17]:
from sklearn.linear_model import RidgeClassifier

ridge_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", RidgeClassifier(random_state=28))
])

ridge_params = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],  # Comparison requirement [cite: 39]
    "tfidf__max_features": [10000,20000, 50000],   # Representation requirement [cite: 38]
    "clf__alpha": [0.1, 1.0, 10.0]           # The "knob" for regularization
}

ridge_grid = run_gridsearch("Ridge Classifier", ridge_pipeline, ridge_params)

Fitting 3 folds for each of 18 candidates, totalling 54 fits

==== Ridge Classifier DETAILED FOLD RESULTS ====
                                                                          params  split0_test_Accuracy  split1_test_Accuracy  split2_test_Accuracy  mean_test_Accuracy  split0_test_MacroF1  split1_test_MacroF1  split2_test_MacroF1  mean_test_MacroF1
 {'clf__alpha': 1.0, 'tfidf__max_features': 50000, 'tfidf__ngram_range': (1, 2)}              0.642354              0.640563              0.640708            0.641208             0.476540             0.475902             0.479258           0.477233
 {'clf__alpha': 0.1, 'tfidf__max_features': 20000, 'tfidf__ngram_range': (1, 2)}              0.637708              0.635802              0.637281            0.636931             0.473894             0.472687             0.478457           0.475013
 {'clf__alpha': 1.0, 'tfidf__max_features': 20000, 'tfidf__ngram_range': (1, 2)}              0.645969              0.643875              0.64

In [10]:
import pandas as pd
from sklearn.model_selection import GridSearchCV

def run_gridsearch(name, pipeline, params):
    # 1. Define multiple metrics
    scoring = {
        'Accuracy': 'accuracy',
        'MacroF1': 'f1_macro'
    }

    grid_st = GridSearchCV(
        pipeline,
        params,
        cv=3,
        n_jobs=-1,
        verbose=1,
        scoring=scoring,
        refit='MacroF1',
        return_train_score=False
    )

    grid_st.fit(x, Y)

    # 2. Extract detailed results into a table
    results_df = pd.DataFrame(grid_st.cv_results_)

    # Select columns for the 3 folds and the means
    columns_to_show = ['params']
    # Add Accuracy columns
    columns_to_show += ['split0_test_Accuracy', 'split1_test_Accuracy', 'split2_test_Accuracy', 'mean_test_Accuracy']
    # Add F1 columns
    columns_to_show += ['split0_test_MacroF1', 'split1_test_MacroF1', 'split2_test_MacroF1', 'mean_test_MacroF1']

    report_table = results_df[columns_to_show].sort_values('mean_test_MacroF1', ascending=False)

    print(f"\n==== {name} DETAILED FOLD RESULTS ====")
    print(report_table.to_string(index=False))

    # 3. Print the absolute bests
    best_f1_idx = results_df['mean_test_MacroF1'].idxmax()
    best_acc_idx = results_df['mean_test_Accuracy'].idxmax()

    print(f"\n--- {name} Summary ---")
    print(f"Best Params (for F1): {grid_st.best_params_}")
    print(f"Best Mean Macro-F1: {results_df.loc[best_f1_idx, 'mean_test_MacroF1']:.4f}")
    print(f"Best Mean Accuracy: {results_df.loc[best_acc_idx, 'mean_test_Accuracy']:.4f}")

    return grid_st

In [11]:
nb_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", MultinomialNB())
])

nb_params = {
    "clf__alpha": [0.1, 0.5, 1.0],
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__max_features": [10000, 20000, 50000]
}

nb_grid = run_gridsearch(
    "Naive Bayes",
    nb_pipeline,
    nb_params
)

Fitting 3 folds for each of 18 candidates, totalling 54 fits

==== Naive Bayes DETAILED FOLD RESULTS ====
                                                                         params  split0_test_Accuracy  split1_test_Accuracy  split2_test_Accuracy  mean_test_Accuracy  split0_test_MacroF1  split1_test_MacroF1  split2_test_MacroF1  mean_test_MacroF1
{'clf__alpha': 0.1, 'tfidf__max_features': 50000, 'tfidf__ngram_range': (1, 2)}              0.635104              0.634896              0.633302            0.634434             0.462244             0.462969             0.462135           0.462449
{'clf__alpha': 0.1, 'tfidf__max_features': 20000, 'tfidf__ngram_range': (1, 2)}              0.633677              0.633135              0.631708            0.632840             0.452543             0.454890             0.453445           0.453626
{'clf__alpha': 0.5, 'tfidf__max_features': 20000, 'tfidf__ngram_range': (1, 2)}              0.634208              0.633521              0.631927     

In [12]:
lr_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LogisticRegression(max_iter=1000))
])

lr_params = {
    "clf__C": [0.5, 1, 3],
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__max_features": [10000, 20000, 50000]
}

lr_grid = run_gridsearch(
    "Logistic Regression",
    lr_pipeline,
    lr_params
)

Fitting 3 folds for each of 18 candidates, totalling 54 fits

==== Logistic Regression DETAILED FOLD RESULTS ====
                                                                     params  split0_test_Accuracy  split1_test_Accuracy  split2_test_Accuracy  mean_test_Accuracy  split0_test_MacroF1  split1_test_MacroF1  split2_test_MacroF1  mean_test_MacroF1
  {'clf__C': 3, 'tfidf__max_features': 20000, 'tfidf__ngram_range': (1, 2)}              0.648135              0.647312              0.648052            0.647833             0.508969             0.509497             0.512469           0.510312
  {'clf__C': 3, 'tfidf__max_features': 10000, 'tfidf__ngram_range': (1, 2)}              0.650583              0.650031              0.649719            0.650111             0.509923             0.508295             0.512018           0.510079
  {'clf__C': 3, 'tfidf__max_features': 50000, 'tfidf__ngram_range': (1, 2)}              0.645531              0.644292              0.644469            0

In [13]:
svm_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1,2),
        max_features=50000
    )),
    ("clf", LinearSVC())
])

svm_params = {
    "clf__C": [0.5, 1, 2],
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__max_features": [10000, 20000, 50000]
}

svm_grid = run_gridsearch(
    "SVM",
    svm_pipeline,
    svm_params
)

Fitting 3 folds for each of 18 candidates, totalling 54 fits

==== SVM DETAILED FOLD RESULTS ====
                                                                     params  split0_test_Accuracy  split1_test_Accuracy  split2_test_Accuracy  mean_test_Accuracy  split0_test_MacroF1  split1_test_MacroF1  split2_test_MacroF1  mean_test_MacroF1
{'clf__C': 0.5, 'tfidf__max_features': 50000, 'tfidf__ngram_range': (1, 2)}              0.642490              0.641990              0.641740            0.642073             0.488442             0.488397             0.490430           0.489089
  {'clf__C': 1, 'tfidf__max_features': 20000, 'tfidf__ngram_range': (1, 2)}              0.641146              0.640458              0.640687            0.640764             0.487638             0.487635             0.491639           0.488971
  {'clf__C': 2, 'tfidf__max_features': 20000, 'tfidf__ngram_range': (1, 2)}              0.634854              0.633469              0.634781            0.634368         

In [14]:
from sklearn.linear_model import SGDClassifier

sgd_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()), # Parameters handled by grid
    ("clf", SGDClassifier(max_iter=1000, tol=1e-3, random_state=25))
])

sgd_params = {
    "clf__alpha": [0.0001, 0.001, 0.01],
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__max_features": [10000, 20000, 50000]
}

sgd_grid = run_gridsearch("SGD Classifier", sgd_pipeline, sgd_params)

Fitting 3 folds for each of 18 candidates, totalling 54 fits

==== SGD Classifier DETAILED FOLD RESULTS ====
                                                                            params  split0_test_Accuracy  split1_test_Accuracy  split2_test_Accuracy  mean_test_Accuracy  split0_test_MacroF1  split1_test_MacroF1  split2_test_MacroF1  mean_test_MacroF1
{'clf__alpha': 0.0001, 'tfidf__max_features': 10000, 'tfidf__ngram_range': (1, 2)}              0.634062              0.633708              0.633646            0.633806             0.402792             0.402921             0.405125           0.403613
{'clf__alpha': 0.0001, 'tfidf__max_features': 20000, 'tfidf__ngram_range': (1, 2)}              0.633542              0.632656              0.632594            0.632931             0.399818             0.400024             0.401631           0.400491
{'clf__alpha': 0.0001, 'tfidf__max_features': 10000, 'tfidf__ngram_range': (1, 1)}              0.631469              0.629448            

In [15]:
from sklearn.linear_model import RidgeClassifier

ridge_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", RidgeClassifier(random_state=28))
])

ridge_params = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__max_features": [10000,20000, 50000],
    "clf__alpha": [0.1, 1.0, 10.0]
}

ridge_grid = run_gridsearch("Ridge Classifier", ridge_pipeline, ridge_params)

Fitting 3 folds for each of 18 candidates, totalling 54 fits

==== Ridge Classifier DETAILED FOLD RESULTS ====
                                                                          params  split0_test_Accuracy  split1_test_Accuracy  split2_test_Accuracy  mean_test_Accuracy  split0_test_MacroF1  split1_test_MacroF1  split2_test_MacroF1  mean_test_MacroF1
 {'clf__alpha': 1.0, 'tfidf__max_features': 50000, 'tfidf__ngram_range': (1, 2)}              0.639781              0.638802              0.639448            0.639344             0.474467             0.473691             0.477435           0.475197
 {'clf__alpha': 0.1, 'tfidf__max_features': 20000, 'tfidf__ngram_range': (1, 2)}              0.637323              0.634938              0.636135            0.636132             0.473857             0.471613             0.477027           0.474166
 {'clf__alpha': 1.0, 'tfidf__max_features': 20000, 'tfidf__ngram_range': (1, 2)}              0.645229              0.643594              0.64

#Deep Learning models

In [26]:
train['rating'] = train['rating'] - 1

In [27]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Use the ORIGINAL review text here, not stemmed/lemmatized!
max_words = 10000
max_len = 100 # Max words per review

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(train['text']) # Assuming 'text' is your raw column

sequences = tokenizer.texts_to_sequences(train['text'])
X_seq = pad_sequences(sequences, maxlen=max_len)
y_seq = train['rating'].values # Use the 0-4 labels you fixed earlier

In [28]:
# Create a smaller subset for the BiLSTM (e.g., 50,000 rows)
train_subset = train.sample(n=57000, random_state=20).copy()

# Now use this subset for your Tokenizer and X_seq/y_seq
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(train_subset['text'])

sequences = tokenizer.texts_to_sequences(train_subset['text'])
X_seq_sub = pad_sequences(sequences, maxlen=max_len)
y_seq_sub = train_subset['rating'].values

In [29]:
import pandas as pd
from sklearn.model_selection import GridSearchCV

def run_gridsearch(name, pipeline, params):
    # 1. Define multiple metrics
    scoring = {
        'Accuracy': 'accuracy',
        'MacroF1': 'f1_macro'
    }

    grid = GridSearchCV(
        pipeline,
        params,
        cv=3,
        n_jobs=-1,
        verbose=1,
        scoring=scoring,
        refit='MacroF1',
        return_train_score=False
    )

    grid.fit(X_seq_sub, y_seq_sub)

    # 2. Extract detailed results into a table
    results_df = pd.DataFrame(grid.cv_results_)

    # Select columns for the 3 folds and the means
    columns_to_show = ['params']
    # Add Accuracy columns
    columns_to_show += ['split0_test_Accuracy', 'split1_test_Accuracy', 'split2_test_Accuracy', 'mean_test_Accuracy']
    # Add F1 columns
    columns_to_show += ['split0_test_MacroF1', 'split1_test_MacroF1', 'split2_test_MacroF1', 'mean_test_MacroF1']

    report_table = results_df[columns_to_show].sort_values('mean_test_MacroF1', ascending=False)

    print(f"\n==== {name} DETAILED FOLD RESULTS ====")
    print(report_table.to_string(index=False))

    # 3. Print the absolute bests
    best_f1_idx = results_df['mean_test_MacroF1'].idxmax()
    best_acc_idx = results_df['mean_test_Accuracy'].idxmax()

    print(f"\n--- {name} Summary ---")
    print(f"Best Params (for F1): {grid.best_params_}")
    print(f"Best Mean Macro-F1: {results_df.loc[best_f1_idx, 'mean_test_MacroF1']:.4f}")
    print(f"Best Mean Accuracy: {results_df.loc[best_acc_idx, 'mean_test_Accuracy']:.4f}")

    return grid

In [30]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout, Input


def create_bilstm_model(lstm_units=64, dropout_rate=0.2, embedding_dim=100):

    model = Sequential([
        Input(shape=(max_len,)),   # ✅ VERY IMPORTANT

        Embedding(
            input_dim=max_words,
            output_dim=embedding_dim
        ),

        Bidirectional(LSTM(lstm_units)),

        Dropout(dropout_rate),

        Dense(32, activation="relu"),

        Dense(5, activation="softmax")
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

from scikeras.wrappers import KerasClassifier

# Wrap for Scikit-Learn
bilstm_clf = KerasClassifier(
    model=create_bilstm_model,
    verbose=0
)

# Parameters to tune for your "Critical Analysis" [cite: 70]
bilstm_params = {
    "model__lstm_units": [32],
    "model__embedding_dim": [100],
    "batch_size": [64],
    "epochs": [10]
}

bilstm_grid = run_gridsearch("BiLSTM", bilstm_clf, bilstm_params)

Fitting 3 folds for each of 1 candidates, totalling 3 fits

==== BiLSTM DETAILED FOLD RESULTS ====
                                                                                params  split0_test_Accuracy  split1_test_Accuracy  split2_test_Accuracy  mean_test_Accuracy  split0_test_MacroF1  split1_test_MacroF1  split2_test_MacroF1  mean_test_MacroF1
{'batch_size': 64, 'epochs': 10, 'model__embedding_dim': 100, 'model__lstm_units': 32}              0.602421              0.586579              0.590842            0.593281             0.479713              0.48224             0.480069           0.480674

--- BiLSTM Summary ---
Best Params (for F1): {'batch_size': 64, 'epochs': 10, 'model__embedding_dim': 100, 'model__lstm_units': 32}
Best Mean Macro-F1: 0.4807
Best Mean Accuracy: 0.5933


In [32]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout, Input


def create_bilstm_model(lstm_units=64, dropout_rate=0.2, embedding_dim=100):

    model = Sequential([
        Input(shape=(max_len,)),   # ✅ VERY IMPORTANT

        Embedding(
            input_dim=max_words,
            output_dim=embedding_dim
        ),

        Bidirectional(LSTM(lstm_units)),

        Dropout(dropout_rate),

        Dense(32, activation="relu"),

        Dense(5, activation="softmax")
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

from scikeras.wrappers import KerasClassifier

# Wrap for Scikit-Learn
bilstm_clf = KerasClassifier(
    model=create_bilstm_model,
    verbose=0
)

# Parameters to tune for your "Critical Analysis" [cite: 70]
bilstm_params = {
    "model__lstm_units": [32],
    "model__embedding_dim": [100],
    "batch_size": [64],
    "epochs": [5]
}

bilstm_grid = run_gridsearch("BiLSTM", bilstm_clf, bilstm_params)

Fitting 3 folds for each of 1 candidates, totalling 3 fits

==== BiLSTM DETAILED FOLD RESULTS ====
                                                                               params  split0_test_Accuracy  split1_test_Accuracy  split2_test_Accuracy  mean_test_Accuracy  split0_test_MacroF1  split1_test_MacroF1  split2_test_MacroF1  mean_test_MacroF1
{'batch_size': 64, 'epochs': 5, 'model__embedding_dim': 100, 'model__lstm_units': 32}              0.619579              0.618211              0.603632            0.613807             0.488252             0.499946             0.499348           0.495849

--- BiLSTM Summary ---
Best Params (for F1): {'batch_size': 64, 'epochs': 5, 'model__embedding_dim': 100, 'model__lstm_units': 32}
Best Mean Macro-F1: 0.4958
Best Mean Accuracy: 0.6138


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout, Input


def create_bilstm_model(lstm_units=64, dropout_rate=0.2, embedding_dim=100):

    model = Sequential([
        Input(shape=(max_len,)),   # ✅ VERY IMPORTANT

        Embedding(
            input_dim=max_words,
            output_dim=embedding_dim
        ),

        Bidirectional(LSTM(lstm_units)),

        Dropout(dropout_rate),

        Dense(32, activation="relu"),

        Dense(5, activation="softmax")
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

from scikeras.wrappers import KerasClassifier

# Wrap for Scikit-Learn
bilstm_clf = KerasClassifier(
    model=create_bilstm_model,
    verbose=0
)

# Parameters to tune for your "Critical Analysis" [cite: 70]
bilstm_params = {
    "model__lstm_units": [64],
    "model__embedding_dim": [100],
    "batch_size": [64],
    "epochs": [5]
}

bilstm_grid = run_gridsearch("BiLSTM", bilstm_clf, bilstm_params)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout, Input


def create_bilstm_model(lstm_units=64, dropout_rate=0.2, embedding_dim=100):

    model = Sequential([
        Input(shape=(max_len,)),   # ✅ VERY IMPORTANT

        Embedding(
            input_dim=max_words,
            output_dim=embedding_dim
        ),

        Bidirectional(LSTM(lstm_units)),

        Dropout(dropout_rate),

        Dense(32, activation="relu"),

        Dense(5, activation="softmax")
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

from scikeras.wrappers import KerasClassifier

# Wrap for Scikit-Learn
bilstm_clf = KerasClassifier(
    model=create_bilstm_model,
    verbose=0
)

# Parameters to tune for your "Critical Analysis" [cite: 70]
bilstm_params = {
    "model__lstm_units": [128],
    "model__embedding_dim": [100],
    "batch_size": [64],
    "epochs": [5]
}

bilstm_grid = run_gridsearch("BiLSTM", bilstm_clf, bilstm_params)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout, Input


def create_bilstm_model(lstm_units=64, dropout_rate=0.2, embedding_dim=100):

    model = Sequential([
        Input(shape=(max_len,)),   # ✅ VERY IMPORTANT

        Embedding(
            input_dim=max_words,
            output_dim=embedding_dim
        ),

        Bidirectional(LSTM(lstm_units)),

        Dropout(dropout_rate),

        Dense(32, activation="relu"),

        Dense(5, activation="softmax")
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

from scikeras.wrappers import KerasClassifier

# Wrap for Scikit-Learn
bilstm_clf = KerasClassifier(
    model=create_bilstm_model,
    verbose=0
)

# Parameters to tune for your "Critical Analysis" [cite: 70]
bilstm_params = {
    "model__lstm_units": [32],
    "model__embedding_dim": [200],
    "batch_size": [64],
    "epochs": [5]
}

bilstm_grid = run_gridsearch("BiLSTM", bilstm_clf, bilstm_params)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout, Input


def create_bilstm_model(lstm_units=64, dropout_rate=0.2, embedding_dim=100):

    model = Sequential([
        Input(shape=(max_len,)),   # ✅ VERY IMPORTANT

        Embedding(
            input_dim=max_words,
            output_dim=embedding_dim
        ),

        Bidirectional(LSTM(lstm_units)),

        Dropout(dropout_rate),

        Dense(32, activation="relu"),

        Dense(5, activation="softmax")
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

from scikeras.wrappers import KerasClassifier

# Wrap for Scikit-Learn
bilstm_clf = KerasClassifier(
    model=create_bilstm_model,
    verbose=0
)

# Parameters to tune for your "Critical Analysis" [cite: 70]
bilstm_params = {
    "model__lstm_units": [64],
    "model__embedding_dim": [200],
    "batch_size": [64],
    "epochs": [5]
}

bilstm_grid = run_gridsearch("BiLSTM", bilstm_clf, bilstm_params)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout, Input


def create_bilstm_model(lstm_units=64, dropout_rate=0.2, embedding_dim=100):

    model = Sequential([
        Input(shape=(max_len,)),   # ✅ VERY IMPORTANT

        Embedding(
            input_dim=max_words,
            output_dim=embedding_dim
        ),

        Bidirectional(LSTM(lstm_units)),

        Dropout(dropout_rate),

        Dense(32, activation="relu"),

        Dense(5, activation="softmax")
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

from scikeras.wrappers import KerasClassifier

# Wrap for Scikit-Learn
bilstm_clf = KerasClassifier(
    model=create_bilstm_model,
    verbose=0
)

# Parameters to tune for your "Critical Analysis" [cite: 70]
bilstm_params = {
    "model__lstm_units": [128],
    "model__embedding_dim": [200],
    "batch_size": [64],
    "epochs": [5]
}

bilstm_grid = run_gridsearch("BiLSTM", bilstm_clf, bilstm_params)